# Part 4: Map Validation, Ground-Truth Overlay & Visualization

This notebook covers the final stage of validation and visualization:
1. **Load Validation Coordinates** (longitude, latitude for bee farms)
2. **Reproject Coordinates** & resolve overlapping duplicate locations to avoid overlapping map numbers
3. **Extract suitability classes** at ground-truth sites
4. **Render and Export Final Maps** (individual criterion maps, continuous score map, and 5-class final suitability map with validation points overlay)
5. **Save validation logs** as CSVs

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

sys.path.append(os.path.abspath('..'))

### 1. Load and Align Validation Ground-Truth coordinates
We group overlapping coordinates and assign display numbers dynamically using our validation suite module.

In [ ]:
from src.config import validation_data
from src.validation_plot import process_validation_points

# Process and extract classes
validation_inside, validation_plot = process_validation_points(
    validation_data, meta, study_area_mask, final_cat
)

print(f"Total validation records processed: {len(validation_inside)}")
print(f"Unique visible numbered map locations: {len(validation_plot)}")
print(validation_plot[['Display_No', 'Original_Point_Nos', 'Place_Organization', 'Suitability_Class']])

### 2. Plotting final 5-Class Map with Validation points
We render the final map overlaying validation sites, custom labels, custom legend, graphic scale bar, and North arrow.

In [ ]:
from src.validation_plot import (
    apply_study_mask, make_masked_cat_cmap, add_north_arrow, add_text_scale_bar
)
from src.config import cat_colors, labels, PIXEL_AREA_M2

# Create masked colormap and apply mask to final category map
cmap_cat = make_masked_cat_cmap(cat_colors)
bounds = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]
norm_cat = mcolors.BoundaryNorm(bounds, cmap_cat.N)

cat_plot = apply_study_mask(final_cat.astype(float), study_area_mask)

fig, ax = plt.subplots(figsize=(14, 12))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# Render suitability raster
ax.imshow(cat_plot, cmap=cmap_cat, norm=norm_cat, interpolation='none')

# Scatter plot unique validation locations
ax.scatter(
    validation_plot['plot_col'],
    validation_plot['plot_row'],
    s=45,
    marker='o',
    facecolor='cyan',
    edgecolor='black',
    linewidth=0.8,
    zorder=30,
    label='Beehive / Bee Validation Points'
)

# Label unique validation locations
for _, point in validation_plot.iterrows():
    ax.text(
        point['plot_col'] + 8,
        point['plot_row'] - 8,
        str(int(point['Display_No'])),
        fontsize=8,
        fontweight='bold',
        color='black',
        bbox=dict(boxstyle='circle,pad=0.18', facecolor='white', edgecolor='black', linewidth=0.4, alpha=0.85),
        zorder=31
    )

# Build legend
patches = [
    mpatches.Patch(
        facecolor=cat_colors[i],
        edgecolor='#444444',
        linewidth=0.6,
        label=f"Class {i+1} — {labels[i+1]:<10} {final_class_stats[i+1]['pct']:5.1f}% ({final_class_stats[i+1]['area_km2']:.1f} km²)"
    ) for i in range(5)
]
patches.append(plt.Line2D(
    [0], [0], marker='o', color='black', markerfacecolor='cyan', markeredgecolor='black',
    markersize=7, linestyle='None', label='Beehive / Bee Validation Points'
))

ax.legend(
    handles=patches, loc='lower right', fontsize=9,
    title='Suitability Class (% area | km²)', title_fontsize=10,
    framealpha=0.92, facecolor='white', edgecolor='#777777', fancybox=True
)

ax.set_title(
    'Beehive Suitability Map — Kavrepalanchowk District, Nepal\n',
    fontsize=14, fontweight='bold', color='#1F3864', pad=14
)
ax.axis('off')

# Add map utilities
add_north_arrow(ax, x=0.975, y=0.975, size=16)
add_text_scale_bar(ax, x=0.045, y=0.035, width=0.090, tick_height=0.010, fontsize=7)

plt.tight_layout()
output_map_path = '../data/3_validation_map/Beehive_Suitability_5Class_FINAL.png'
plt.savefig(output_map_path, dpi=250, bbox_inches='tight', facecolor='white', transparent=False)
plt.show()
print(f"✓ Validation map successfully saved to: {output_map_path}")

### 3. Save validation tabular logs

In [ ]:
os.makedirs('../data/3_validation_map', exist_ok=True)
validation_all_path = '../data/3_validation_map/Beehive_Validation_Points_All_Extracted_Suitability.csv'
validation_visible_path = '../data/3_validation_map/Beehive_Validation_Points_VISIBLE_Map_Numbering.csv'

validation_inside.drop(columns='geometry').to_csv(validation_all_path, index=False)
validation_plot.to_csv(validation_visible_path, index=False)

print(f"✓ Saved complete logs to: {validation_all_path}")
print(f"✓ Saved visible numbered logs to: {validation_visible_path}")